# ada-nest job console — the ONE Colab notebook for queue work

All GPU workloads run as **queue jobs + packs** on Drive
(`ada-nest/queue/JOBS.py` + `ada-nest/packs/<name>/pack.yml`) — never as
hand-run per-job cells. This notebook is only the operator surface:
run cell 1 once, then start the shared queue through `mcp-colab` or use
cell 2 as a manual fallback.

## How to run a session

1. **Runtime → Change runtime type → T4.** Consumer Colab runtime
   provisioning remains a manual UI action; this controller does not create
   or stop account runtimes.
2. **Run cell 1** — mounts Drive, prints this VM's identity, checks for a
   managed peer before registering, and starts a small Drive-command watcher.
   After `PREFLIGHT-OK`, `mcp-colab` can start the shared queue for this exact
   boot ID without requiring cell 2.
3. **Optional fallback — run cell 2** to start the shared queue manually if
   the MCP command path is unavailable. The queue exists to fill a GPU;
   cell 2 refuses to start on a CPU runtime.

The queue then drains `JOBS.py` (per-runtime warmup → HF-cache restore →
deps → assets → cache save) and every un-done `packs/<name>` — voice,
video, DevNest, eval packs all ride the same runner in the same VM.

## Watching / stopping

- Fleet-side: `rclone cat gdrive:ada-nest/queue/run-<ts>.out.log`,
  `queue/heartbeat`, `queue/done/`, `packs/<name>/done`,
  `queue/runtime-sessions/*.json`, `queue/runner.id`.
- In-notebook diagnostics are safe any time (the runner is detached):
  `!nvidia-smi`, `!df -h`, etc.
- **Stopping:** the queue stops with the VM (Runtime → disconnect) or
  Runtime → interrupt execution. Nothing here ever stops *another*
  session — **Google Runtime > Manage Sessions is the authoritative
  inventory** for notebooks that didn't run cell 1.


### Cell 1 — mount Drive + register this managed runtime

Checks for a fresh peer before registering this VM. If a peer is present,
the cell refuses without leaving a heartbeat record. Otherwise it writes
`queue/runtime-sessions/<boot_id>.json` and refreshes it on a background
heartbeat, then starts a watcher for MCP queue-start requests targeted at
this boot ID. Resolve peers in Runtime > Manage Sessions.


In [ ]:
# ada-nest JOB CONSOLE — CELL 1: mount Drive, identify + register this
# managed runtime, refuse ambiguous duplicate launches. Run ONCE per session,
# before CELL 2. Canonical source: scripts/colab/preflight-cell.py (chaba
# repo); deployed copy lives at ada-nest/queue/preflight-cell.py.
#
# What it does:
#   1. mounts Google Drive at /content/drive
#   2. prints this VM's identity: boot_id, GPU model/VRAM, RAM, Drive state
#   3. registers the VM at queue/runtime-sessions/<boot_id>.json and refreshes it
#      on a background heartbeat while this notebook stays open
#   4. if ANOTHER managed runtime's record is still fresh (<TTL), refuses —
#      a second console session is ambiguous: it may be about to launch its
#      own queue. Resolve via Google Runtime > Manage Sessions (authoritative
#      for ALL sessions incl. uninstrumented notebooks) or re-run after the
#      peer goes stale (~5 min). Deliberate two-VM ops: set
#      ALLOW_SECOND_RUNTIME = True below and re-run — the queue lease then
#      arbitrates writers.
#
# Nothing here stops or kills another session — ever. Unknown/uninstrumented
# sessions are Google's UI problem, not ours.
import contextlib
import glob
import io
import json
import os
import runpy
import sys
import threading
import time
import uuid

Q = os.environ.get('NEST_QUEUE', '/content/drive/MyDrive/ada-nest/queue')
ALLOW_SECOND_RUNTIME = False        # deliberate dual-VM escape hatch only
HB_S = 60                            # registry refresh cadence
CONTROL_POLL_S = 5
CONTROL_TTL_S = 300


def mount_drive():
    if os.environ.get('NEST_ROOT'):
        return 'sandbox NEST_ROOT — skipping drive.mount'
    from google.colab import drive
    drive.mount('/content/drive')
    return 'drive mounted at /content/drive'


def heartbeat_loop(rr):
    while True:
        time.sleep(HB_S)
        try:
            rr.register()
        except Exception:
            pass


def _write_control_ack(q, request_id, target_boot_id, status, message):
    ack_dir = os.path.join(q, 'control', 'acks')
    os.makedirs(ack_dir, exist_ok=True)
    ack_path = os.path.join(ack_dir, f'{request_id}.json')
    tmp = f'{ack_path}.{uuid.uuid4().hex}.tmp'
    with open(tmp, 'w') as f:
        json.dump({
            'request_id': request_id,
            'target_boot_id': target_boot_id,
            'status': status,
            'message': str(message)[-2000:],
            'handled_at': time.time(),
        }, f)
        f.write('\n')
    os.replace(tmp, ack_path)
    return ack_path


def process_control_request(request_path, boot_id, queue_dir=None,
                            launcher_main=None):
    """Handle one managed-runtime command; foreign requests are left alone."""
    q = queue_dir or Q
    request_id = os.path.basename(request_path).removesuffix('.json')
    ack_path = os.path.join(q, 'control', 'acks', f'{request_id}.json')
    if os.path.exists(ack_path):
        return {'status': 'already-acked', 'request_id': request_id}

    try:
        with open(request_path) as f:
            request = json.load(f)
    except Exception as exc:
        return {'status': 'unreadable', 'request_id': request_id,
                'message': str(exc)}
    target = str(request.get('target_boot_id') or '')
    if target != boot_id:
        return {'status': 'not-targeted', 'request_id': request_id}
    if request.get('request_id') != request_id:
        return {'status': 'invalid', 'request_id': request_id,
                'message': 'request id does not match filename'}
    if request.get('confirmed') is not True:
        _write_control_ack(q, request_id, target, 'rejected',
                           'explicit MCP confirmation is required')
        return {'status': 'rejected', 'request_id': request_id}
    try:
        age = time.time() - float(request.get('created_at') or 0)
    except (TypeError, ValueError):
        _write_control_ack(q, request_id, target, 'rejected',
                           'request created_at is invalid')
        return {'status': 'rejected', 'request_id': request_id}
    if age > CONTROL_TTL_S:
        _write_control_ack(q, request_id, target, 'expired',
                           f'request expired ({age:.0f}s old)')
        return {'status': 'expired', 'request_id': request_id}
    if request.get('command') != 'start_queue':
        _write_control_ack(q, request_id, target, 'rejected',
                           f"unsupported command: {request.get('command')!r}")
        return {'status': 'rejected', 'request_id': request_id}

    output = io.StringIO()
    try:
        if launcher_main is None:
            launcher = runpy.run_path(
                os.path.join(q, 'queue-launcher-cell.py'),
                run_name='mcp_queue_launcher')
            launcher_main = launcher['main']
        with contextlib.redirect_stdout(output), contextlib.redirect_stderr(output):
            launcher_main()
        message = output.getvalue()
        status = ('already-running' if 'queue already running' in message
                  else 'started')
    except SystemExit as exc:
        message = output.getvalue() + str(exc)
        status = 'refused'
    except Exception as exc:
        message = output.getvalue() + repr(exc)
        status = 'error'

    _write_control_ack(q, request_id, target, status, message)
    return {'status': status, 'request_id': request_id, 'message': message}


def control_watcher_loop(queue_dir, boot_id):
    """Listen for MCP requests directed at this already-running runtime."""
    requests = os.path.join(queue_dir, 'control', 'requests')
    while True:
        for path in sorted(glob.glob(os.path.join(requests, '*.json'))):
            try:
                process_control_request(path, boot_id, queue_dir=queue_dir)
            except Exception:
                continue
        time.sleep(CONTROL_POLL_S)


def main():
    print(mount_drive())
    sys.path.insert(0, Q)
    import runtime_registry as rr
    fresh = rr.peers()
    if fresh and not ALLOW_SECOND_RUNTIME:
        names = '; '.join(rr.fmt_runtime(p) for p in fresh)
        raise SystemExit(
            'REFUSED: another managed runtime is fresh: ' + names +
            ' — ambiguous duplicate launch. Check Runtime > Manage Sessions; '
            'to run a second VM deliberately set ALLOW_SECOND_RUNTIME = True '
            'in this cell and re-run.')

    rec = rr.register()
    print(f"runtime: boot {rec['boot_id'][:8]}  gpu "
          f"{rec['gpu'] or 'NONE'} {rec['gpu_gb']:.1f}GB  ram {rec['ram_gb']}GB"
          f"  drive_mounted={rec['drive_mounted']}")
    print(f"registered -> {Q}/runtime-sessions/{rec['boot_id']}.json"
          f"  (heartbeat every {HB_S}s, ttl {rr.ttl_s():.0f}s)")
    threading.Thread(target=heartbeat_loop, args=(rr,), daemon=True).start()
    if not os.environ.get('NEST_ROOT'):
        control_boot = rec['boot_id']
        started = globals().setdefault('_NEST_CONTROL_WATCHERS', set())
        if control_boot not in started:
            started.add(control_boot)
            threading.Thread(
                target=control_watcher_loop, args=(Q, control_boot),
                daemon=True).start()
            print('MCP queue-start control ready for this runtime.')

    if fresh:
        print('NOTE: proceeding alongside fresh peer(s) '
              '(ALLOW_SECOND_RUNTIME): ' +
              '; '.join(rr.fmt_runtime(p) for p in fresh))
    old = rr.stale()
    if old:
        print('stale registry records (idle info only): '
              + '; '.join(rr.fmt_runtime(p) for p in old))
    if not rec['gpu']:
        print('NOTE: no GPU on this runtime — cell 2 will refuse to start '
              'the queue (Runtime > Change runtime type > T4).')
    print('PREFLIGHT-OK — run cell 2 to start the shared queue once.')


if __name__ == '__main__':
    main()


### Cell 2 — manual fallback to start the shared queue

Prefer `mcp-colab` `colab_queue_start(boot_id)` after cell 1. Use this cell
only when the MCP control path is unavailable.

Runs the full gate before spawning: Drive mounted, GPU present, no fresh
peer runtime, and runner liveness is **boot-scoped** (`queue/runner.id`
carries `boot_id` — a bare PID on shared Drive is never a process handle,
so a PID from another VM's stale record is never signaled).


In [ ]:
# ada-nest JOB CONSOLE — CELL 2: start the detached shared queue ONCE.
# Canonical source: scripts/colab/queue-launcher-cell.py (chaba repo);
# deployed copy lives at ada-nest/queue/queue-launcher-cell.py.
#
# Starts queue/runner.py DETACHED (own session, output to Drive) so the
# kernel stays free for diagnostics and the queue survives kernel restart.
#
# Gates, in order — any failure prints why and refuses:
#   1. Drive mounted + runner.py present
#   2. GPU detected (CPU runtimes must not start the GPU queue)
#   3. no OTHER managed runtime fresh in queue/runtime-sessions/ (ambiguous duplicate
#      — see Runtime > Manage Sessions; ALLOW_SECOND_RUNTIME to override)
#   4. runner liveness is boot-scoped: queue/runner.id carries
#      {pid,boot_id,worker,epoch}. A pid is only ever signaled when its
#      recorded boot_id equals THIS VM's — a stale Drive record can name a
#      PID that happens to be alive locally on a different VM, and killing
#      it would be a PID-collision kill (OPS-0872 lesson). Foreign/stale
#      records are left alone; a fresh heartbeat from another boot means the
#      queue is already running elsewhere and we refuse.
# Rerun-safe: a live same-VM runner is reported, a stale-heartbeat same-VM
# zombie is reaped, anything else spawns a fresh detached runner.
import subprocess, os, sys, time, json

ALLOW_SECOND_RUNTIME = False        # deliberate dual-VM escape hatch only
REAP_S = 90                          # heartbeat age that makes a live pid a zombie


def _q():
    return os.environ.get('NEST_QUEUE', '/content/drive/MyDrive/ada-nest/queue')


def _boot_id():
    try:
        return open('/proc/sys/kernel/random/boot_id').read().strip()
    except OSError:
        return 'noboot-' + time.strftime('%Y%m%d-%H%M')


def _heartbeat_age(q):
    try:
        return time.time() - float(open(f'{q}/heartbeat').read().split()[0])
    except Exception:
        return float('inf')


def _read_runner_id(q):
    try:
        return json.loads(open(f'{q}/runner.id').read())
    except Exception:
        return None


def _pid_alive(pid):
    """Signal-0 liveness — ONLY ever called on pids whose runner.id boot_id
    already matched THIS VM. Never call on a Drive-recorded foreign pid."""
    try:
        os.kill(int(pid), 0)
        return True
    except (ProcessLookupError, ValueError, PermissionError):
        return False


def runner_state(q=None):
    """(state, detail) —
    'alive-local'      our VM's runner is up and heartbeating
    'zombie-local'     our VM's runner pid lives but heartbeat is stale (reap)
    'alive-elsewhere'  fresh queue heartbeat owned by another boot — refuse
    'free'             no live runner anywhere we can prove
    """
    q = q or _q()
    rid = _read_runner_id(q)
    hb = _heartbeat_age(q)
    me = _boot_id()
    if rid and str(rid.get('boot_id')) == me:
        pid = rid.get('pid')
        if pid and _pid_alive(pid):
            if hb <= REAP_S:
                return 'alive-local', f"pid {pid} heartbeat {hb:.0f}s"
            return 'zombie-local', f"pid {pid} heartbeat {hb:.0f}s stale"
        return 'free', f'same-boot runner.id pid {pid} not alive'
    if rid:
        if hb <= REAP_S:
            return ('alive-elsewhere',
                    f"runner.id names other boot {str(rid.get('boot_id'))[:8]} "
                    f"worker {rid.get('worker')} — queue heartbeat {hb:.0f}s fresh")
        return 'free', (f"foreign runner.id (boot {str(rid.get('boot_id'))[:8]}) "
                        f"stale — never signaling pid {rid.get('pid')}")
    # legacy runner.pid (bare number, no boot attribution): it is NOT a process
    # handle on this VM — never os.kill it. Queue heartbeat is the only signal.
    legacy = os.path.exists(f'{q}/runner.pid')
    if hb <= REAP_S:
        return 'alive-elsewhere', (f'fresh queue heartbeat ({hb:.0f}s) with '
                                   + ('legacy runner.pid' if legacy else 'no runner.id')
                                   + ' — queue active on another runtime')
    return 'free', ('legacy runner.pid' if legacy else 'no runner record') + \
                   f', heartbeat {hb if hb != float("inf") else "absent"}'


def launch_gate(q=None):
    """Preflight + duplicate check. Returns problems list (empty = go)."""
    q = q or _q()
    sys.path.insert(0, q)
    try:
        import runtime_registry as rr
        return rr.launch_gate(allow_second=ALLOW_SECOND_RUNTIME)
    except ImportError:
        problems = []
        if not os.path.exists(f'{q}/runner.py'):
            problems.append(f'{q}/runner.py missing — is Drive mounted?')
        return problems


def reap(pid):
    try:
        os.kill(int(pid), 15)
        time.sleep(3)
        os.kill(int(pid), 9)
    except ProcessLookupError:
        pass


def spawn(q):
    ts = time.strftime('%Y%m%d-%H%M')
    out = open(f'{q}/run-{ts}.out.log', 'ab')
    p = subprocess.Popen([sys.executable, '-u', f'{q}/runner.py'],
                         stdout=out, stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL, start_new_session=True)
    print(f'queue pid {p.pid} -> run-{ts}.out.log')
    return p


def main():
    q = _q()
    problems = launch_gate(q)
    if problems:
        raise SystemExit('QUEUE-START REFUSED:\n - ' + '\n - '.join(problems))

    state, detail = runner_state(q)
    print(f'runner state: {state} ({detail})')
    if state == 'alive-local':
        rid = _read_runner_id(q) or {}
        print(f"queue already running, pid {rid.get('pid')}")
    elif state == 'zombie-local':
        rid = _read_runner_id(q) or {}
        print(f"reaping same-VM zombie pid {rid.get('pid')}")
        reap(rid.get('pid'))
        spawn(q)
    elif state == 'alive-elsewhere':
        raise SystemExit(
            'QUEUE-START REFUSED: ' + detail +
            ' — another runtime is mid-queue. Check Runtime > Manage '
            'Sessions; wait for its heartbeat to go stale or finish there.')
    else:
        spawn(q)

    print('watch from fleet: rclone cat gdrive:ada-nest/queue/run-<ts>.out.log')
    print('  heartbeat: queue/heartbeat | done: queue/done/ | preps: queue/prep-*.log')
    print('  identity:  queue/runner.id | runtime-sessions: queue/runtime-sessions/')
    print('  stop it:   Runtime > interrupt execution (kills this VM only)')


if __name__ == '__main__':
    main()
